# 感知手撕

[NumPy NMS 脚本](nms.py) 按置信度依次保留框，并抑制和当前框 IoU 超过阈值的候选。[PyTorch 笔记](1.%20Non-Maximum%20Supression.ipynb) 展开了两两 IoU 与按类 NMS 的实现。这里用坐标连续的 `[x1, y1, x2, y2]` 单独演示两个框的 IoU。

In [ ]:
def box_iou(a, b):
    def area(box):
        return max(0, box[2] - box[0]) * max(0, box[3] - box[1])

    intersection = area((max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])))
    union = area(a) + area(b) - intersection
    return intersection / union if union else 0.0


assert box_iou((0, 0, 2, 2), (1, 1, 3, 3)) == 1 / 7
assert box_iou((0, 0, 1, 1), (1, 0, 2, 1)) == 0.0

## 二分图带权匹配

跟踪里可将检测框和轨迹分别放在两侧，以 IoU 或其他指标构造代价矩阵。小规模时可用状态压缩 DP 理解全局最优分配。状态记录已占用的列，行号等于已占用列数，时间 O(n × 2ⁿ)。SORT 的线性分配支持矩形矩阵，参见 [跟踪实现](../tracking/sort.py)。

In [ ]:
from functools import cache


def minimum_assignment_cost(cost):
    n = len(cost)

    @cache
    def visit(mask):
        row = mask.bit_count()
        if row == n:
            return 0
        return min(cost[row][col] + visit(mask | 1 << col) for col in range(n) if not mask & (1 << col))

    return visit(0)


assert minimum_assignment_cost([[4, 1, 3], [2, 0, 5], [3, 2, 2]]) == 5
assert minimum_assignment_cost([]) == 0